In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle

In [4]:
data=pd.read_csv('Churn_Modelling.csv')
data= data.drop(['RowNumber','CustomerId','Surname'],axis=1)

label_encoder_gender=LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded=onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df=pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

data=pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)

X=data.drop('Exited',axis=1)
y=data['Exited']

X_train, X_test, y_train, y_test=train_test_split(X,y, test_size=0.2,random_state=42)

scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)


# Save encoders and scaler for later use
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)

with open('onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(onehot_encoder_geo,file)

with open('scaler.pkl','wb')as file:
    pickle.dump(scaler,file)


In [5]:
## Define a function to create the model and try diffrent paramaters (KerasClasifier)
def create_model(neurons=32,layer=1):
    model=Sequential()
    model.add(Dense(neurons,activation='relu',input_shape=(X_train.shape[1],)))

    for _ in range(layer-1):
        model.add(Dense(neurons,activation='relu'))

    model.add(Dense(1,activation='relu'))

    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='adam',loss="binary_crossentropy",metrics=['accuracy'])

    return model

In [11]:
## Create a Keras Classifier
from scikeras.wrappers import KerasClassifier

model = KerasClassifier(
    model=create_model,
    epochs=50,
    batch_size=10,
    verbose=0
)

In [14]:
#Define the grid search parameters
param_grid = {
    "model__neurons": [16, 32],
    "model__layer": [1, 2],
    "epochs": [10]
}


In [15]:
grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    n_jobs=1,
    verbose=2
)

grid_result = grid.fit(X_train, y_train)


Fitting 3 folds for each of 4 candidates, totalling 12 fits
[CV] END .......epochs=10, model__layer=1, model__neurons=16; total time=   5.9s
[CV] END .......epochs=10, model__layer=1, model__neurons=16; total time=   5.8s
[CV] END .......epochs=10, model__layer=1, model__neurons=16; total time=   5.6s
[CV] END .......epochs=10, model__layer=1, model__neurons=32; total time=   5.8s
[CV] END .......epochs=10, model__layer=1, model__neurons=32; total time=   5.6s
[CV] END .......epochs=10, model__layer=1, model__neurons=32; total time=   5.5s
[CV] END .......epochs=10, model__layer=2, model__neurons=16; total time=   6.0s
[CV] END .......epochs=10, model__layer=2, model__neurons=16; total time=   5.9s
[CV] END .......epochs=10, model__layer=2, model__neurons=16; total time=   5.9s
[CV] END .......epochs=10, model__layer=2, model__neurons=32; total time=   5.8s
[CV] END .......epochs=10, model__layer=2, model__neurons=32; total time=   5.9s
[CV] END .......epochs=10, model__layer=2, model_